# LaCAM centralized baseline ( S3) — clean build, Colab **or** Kaggle

Runs the **real LaCAM** solver (Kei18/lacam3) on our den520d saturation densities.

**Design (important):** pogema-benchmark's own pipeline is internally inconsistent —
its env code needs a pogema with `AgentsDensityWrapper` (1.3.x) while its solver needs
a pogema whose observations carry `global_obstacles` (the alpha, which lacks the
wrapper); no pip-installable pogema has both. So we run LaCAM's solver in **our proven
pogema-1.3.0 stack** and inject the 3 global keys LaCAM reads from the grid
(`lacam_run.py` + `GlobalObsWrapper`, both verified locally). Only `liblacam.so` (C++)
and LaCAM's `inference.py` come from pogema-benchmark; it builds the `.so` at import.

Auto-detects Colab/Kaggle. Resumable + persisted. Gate = the smoke test (cell 6).

## 1. Platform + paths

In [ ]:
import os
KAGGLE = os.path.exists('/kaggle')
BASE  = '/kaggle/working' if KAGGLE else '/content'
BENCH = f'{BASE}/pogema-benchmark'
OURS  = f'{BASE}/mapf-deadlock'
print('platform:', 'Kaggle' if KAGGLE else 'Colab', '| BASE =', BASE)

## 2. Build tools + clone (pogema-benchmark RECURSIVE for lacam3; our repo for the map + runner)

In [ ]:
!apt-get -qq update && apt-get -qq install -y cmake build-essential >/dev/null
if not os.path.exists(BENCH):
    !git clone --recursive -q https://github.com/Cognitive-AI-Systems/pogema-benchmark.git {BENCH}
!cd {BENCH} && git submodule update --init --recursive
if not os.path.exists(OURS):
    !git clone -q --depth 1 https://github.com/tay805/mapf-deadlock.git {OURS}
else:
    !cd {OURS} && git pull -q
print('lacam3 present:', os.path.isdir(f'{BENCH}/algorithms/lacam/lacam3'))

## 3. py3.10 conda env + our PROVEN stack
Native Python on Colab/Kaggle is 3.12 (no wheels for the stack), so use py3.10 conda.
This is the same env Follower runs in — known to work — NOT pogema-benchmark's drifted
pins. LaCAM's `inference.py` imports cleanly here (verified).

In [ ]:
import subprocess
# py3.10 conda (Colab/Kaggle native is 3.12). SELF-HEALING: if a 'follower' env
# already exists but has the wrong pogema (e.g. a polluted earlier attempt), wipe and
# recreate it clean. Otherwise reuse it. Then install our proven stack.
if KAGGLE:
    if not os.path.exists('/kaggle/working/miniforge'):
        !wget -qO /tmp/mf.sh https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
        !bash /tmp/mf.sh -b -p /kaggle/working/miniforge >/dev/null
    CONDA = '/kaggle/working/miniforge/bin/conda'
else:
    try:
        import condacolab; condacolab.check()
    except Exception:
        !pip install -q condacolab
        import condacolab; condacolab.install()   # restarts kernel; re-run this cell after
    CONDA = 'conda'
exists = 'follower' in subprocess.run([CONDA,'env','list'],capture_output=True,text=True).stdout
if exists:
    v = subprocess.run([CONDA,'run','-n','follower','python','-c','import pogema;print(pogema.__version__)'],
                       capture_output=True, text=True).stdout.strip()
    if v != '1.3.0':
        print(f'follower env has pogema={v!r} -> recreating clean'); exists = False
        subprocess.run([CONDA,'env','remove','-n','follower','-y'])
if not exists:
    !{CONDA} create -y -n follower python=3.10 >/dev/null
!{CONDA} run -n follower pip install -q pogema==1.3.0 pogema-toolbox==0.1.0 'numpy<=1.23.1' 'pandas<=1.4'
PYRUN = f'{CONDA} run --no-capture-output -n follower python'
print('PYRUN =', PYRUN)

## 4. Put our LaCAM runner next to LaCAM's inference

In [ ]:
import shutil
shutil.copy(f'{OURS}/lacam_run.py', f'{BENCH}/algorithms/lacam_run.py')
MAPS = f'{OURS}/env/test-maps.yaml'
if KAGGLE:
    RESULTS = '/kaggle/working/mapf-deadlock-results'
else:
    try:
        from google.colab import drive; drive.mount('/content/gdrive')
        RESULTS = '/content/gdrive/MyDrive/mapf-deadlock-results'   # adjust to your Drive path
    except Exception:
        RESULTS = f'{BASE}/mapf-deadlock-results'
os.makedirs(RESULTS, exist_ok=True)
print('runner copied; RESULTS =', RESULTS)

## 5. Smoke test — 128 agents, seed 0, 128 steps  *(gate)*
The FIRST run builds `liblacam.so` at import (cmake; ~1 min). Must print a throughput.

In [ ]:
import glob, json, subprocess
r = subprocess.run(PYRUN.split() + ['lacam_run.py','128','0','/tmp/lacam_smoke','128',MAPS],
                   cwd=f'{BENCH}/algorithms')
j = glob.glob('/tmp/lacam_smoke/*.json')
print('files:', j)
assert j, f'FAILED rc={r.returncode} — paste the error above'
print('THROUGHPUT:', json.load(open(j[0]))[0]['metrics']['avg_throughput'])

## 6. Full sweep — den520d 128–640 × 3 seeds  *(resumable, persisted)*

In [ ]:
import subprocess, time
RES = f'{RESULTS}/lacam-sat'
for N in [128, 256, 384, 512, 640]:
    for k in [0, 1, 2]:
        out = f'{RES}/n{N}_s{k}'
        if os.path.exists(f'{out}/LaCAM.json'):
            print(f'n{N} s{k}: skip', flush=True); continue
        print(f'=== n{N} s{k} ===', flush=True); t0 = time.time()
        subprocess.run(PYRUN.split() + ['lacam_run.py', str(N), str(k), out, '512', MAPS],
                       cwd=f'{BENCH}/algorithms')
        ok = os.path.exists(f'{out}/LaCAM.json')
        print('   ', 'OK' if ok else 'FAIL', f'{(time.time()-t0)/60:.1f} min', flush=True)
print('SWEEP DONE')

## 7. LaCAM vs Follower

In [ ]:
import glob, json
from collections import defaultdict
agg = defaultdict(list)
for fp in glob.glob(f'{RESULTS}/lacam-sat/n*_s*/LaCAM.json'):
    for r in json.load(open(fp)):
        agg[r['env_grid_search']['num_agents']].append(r['metrics']['avg_throughput'])
ref = {128:2.23, 256:2.32, 384:1.92, 512:1.56, 640:1.06}
print(f"{'agents':>6}{'LaCAM':>9}{'n':>3}{'Follower':>10}")
for n in sorted(agg):
    x = agg[n]; print(f"{n:>6}{sum(x)/len(x):>9.3f}{len(x):>3}{ref.get(n,0):>10.2f}")